<a href="https://colab.research.google.com/github/iamsohk/2025xmas/blob/main/all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime as dt
import os
import warnings

# 隱藏非必要警告，保持介面乾淨
warnings.simplefilter(action='ignore', category=FutureWarning)

# ==========================
# 1. 股票宇宙 (加入部分你關注嘅類別)
# ==========================
UNIVERSE = [
    "AAPL","MSFT","NVDA","AMZN","GOOGL","META","TSLA",
    "AVGO","AMD","QCOM","TXN","INTC","IBM","ORCL","CRM","ADBE",
    "MU","LRCX","AMAT","KLAC","ADI","NXPI","MRVL","ARM","SMH",
    "NOW","SNOW","PANW","CRWD","DDOG","MDB","NET","TEAM","PLTR",
    "JPM","BAC","V","MA","WFC","MS","GS","BRK-B","AXP","BLK",
    "LLY","UNH","JNJ","ABBV","MRK","TMO","ISRG","VRTX",
    "COST","WMT","HD","PG","KO","PEP","MCD","SBUX","DIS",
    "CAT","GE","HON","WM","RSG",
    "COIN","MSTR","MARA","RIOT", # 加密貨幣相關
    "SPY","QQQ", "IWM"
]

# ==========================
# 2. 工具函數（含 retry 及格式清洗）
# ==========================
def load(t, period="5y"):
    """安全下載數據"""
    for _ in range(3):
        try:
            # 確保下載格式統一
            df = yf.download(t, period=period, progress=False, auto_adjust=False, multi_level_index=False)

            if df.empty:
                continue

            # 確保欄位名乾淨 (處理 yfinance 版本差異)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.droplevel(1)

            # 修正 Adj Close
            if "Adj Close" in df.columns:
                df["Close"] = df["Adj Close"]

            # 確保索引係 Datetime
            df.index = pd.to_datetime(df.index)

            return df
        except Exception:
            continue
    return pd.DataFrame()

def atr(df, n=14):
    h, l, c = df["High"], df["Low"], df["Close"]
    prev = c.shift(1)
    tr = pd.concat([(h-l).abs(), (h-prev).abs(), (l-prev).abs()], axis=1).max(axis=1)
    return tr.rolling(n).mean()

# ==========================
# 3. 大市濾網 (Risk-On / Risk-Off)
# ==========================
def market_ok():
    q = load("QQQ", "1y")
    s = load("SPY", "1y")
    v = load("^VIX", "1y")

    if q.empty or s.empty:
        return False, "⚠️ 數據連接失敗，請檢查網絡"

    v_close = v["Close"].iloc[-1] if not v.empty else 30

    q["MA200"] = q["Close"].rolling(200).mean()
    s["MA50"] = s["Close"].rolling(50).mean()

    # 判斷邏輯：QQQ 站上年線 + SPY 站上季線 + VIX 低於 22
    risk_on = (q["Close"].iloc[-1] > q["MA200"].iloc[-1]) and \
              (s["Close"].iloc[-1] > s["MA50"].iloc[-1]) and \
              (v_close < 22)

    if risk_on:
        return True, "🟢 Risk-On（大市健康，適合操作）"

    return False, f"🔴 Risk-Off（大市轉弱 VIX:{v_close:.2f}，建議觀望）"

# ==========================
# 4. 三大策略 (針對 ETF 及大型股優化)
# ==========================
def sig_drop_ma20(df):
    """回調至 MA20 策略"""
    df["MA20"] = df["Close"].rolling(20).mean()
    # 單日跌幅夠深 或 連續回調
    drop = (df["Close"] / df["Close"].shift(1) - 1) < -0.03
    recent = drop.rolling(3).max().fillna(0).astype(bool)
    # 價格撐在 MA20 之上
    support = (df["Low"] <= df["MA20"]) & (df["Close"] > df["MA20"])
    return recent & support

def sig_atr_panic(df):
    """恐慌超跌反彈"""
    df["ATR"] = atr(df)
    prev_range = df["High"].shift(1) - df["Low"].shift(1)
    # 昨日波幅大過 2倍 ATR (恐慌)
    panic = prev_range > df["ATR"].shift(1) * 2
    # 今日收市反彈過昨日高位
    rebound = df["Close"] > df["High"].shift(1)
    return panic & rebound

def sig_vcp(df):
    """簡易版 VCP 強勢股篩選"""
    close = df["Close"]
    vol = df["Volume"]
    ma50 = close.rolling(50).mean()
    ma150 = close.rolling(150).mean()
    ma200 = close.rolling(200).mean()
    high_250 = close.rolling(250).max()
    low_250 = close.rolling(250).min()

    std10 = close.rolling(10).std()
    std50 = close.rolling(50).std()

    # 趨勢過濾
    trend = (close > ma50) & (ma50 > ma150) & (ma150 > ma200)
    trend &= (close > low_250 * 1.3) & (close > high_250 * 0.75)

    # 波動收縮 (Volatility Contraction)
    shrink = std10 < std50 * 0.6

    # 成交量乾涸 (Dry Up)
    dry = vol.rolling(5).mean() < vol.rolling(50).mean()

    return trend & shrink & dry

# ==========================
# 5. 快速回測 (固定 10% 止蝕 / 20日持有)
# ==========================
def backtest(df, sig):
    idxs = sig[sig].index
    if len(idxs) == 0:
        return 0, 0

    wins = 0; total = 0
    for d in idxs:
        try:
            i = df.index.get_loc(d)
        except:
            continue

        if i + 20 >= len(df):  # 數據不足 20 日唔計
            continue

        entry = df["Close"].iloc[i]
        stop = entry * 0.9 # 10% 止蝕
        window = df.iloc[i+1 : i+21]

        # 檢查期間有無觸發止蝕
        if window["Low"].min() < stop:
            total += 1; continue

        exitp = df["Close"].iloc[i+20]
        if exitp > entry:
            wins += 1

        total += 1

    win_rate = (wins / total * 100) if total > 0 else 0
    return win_rate, total

# ==========================
# 6. 主系統執行
# ==========================
def run():
    print("-" * 30)
    print("🤖 蘇蘇投資助理 - 每日掃描")
    print("-" * 30)

    print("\n🔍【大市檢查】")
    ok, msg = market_ok()
    print(msg)

    if not ok:
        print("\n🛑 安全起見，今日建議只看不動 / 整理 Watchlist。")
        return

    print("\n🚀 正在掃描 (條件: 勝率≥65% & 次數≥5)... 請稍候...")
    out = []

    for t in UNIVERSE:
        df = load(t)
        if df.empty or len(df) < 260:
            continue

        strategies = [
            ("🔥 VCP形態", sig_vcp(df)),
            ("📉 回調MA20", sig_drop_ma20(df)),
            ("⚡ ATR恐慌", sig_atr_panic(df)),
        ]

        for name, sig in strategies:
            # 檢查今日有無訊號
            if not sig.iloc[-1]:
                continue

            # 有訊號先做回測
            win_rate, count = backtest(df, sig)

            if win_rate >= 65 and count >= 5:
                price = df["Close"].iloc[-1]
                ma20 = df["Close"].rolling(20).mean().iloc[-1]
                atr_val = atr(df).iloc[-1]
                atr_stop = price - atr_val * 2

                out.append([t, name, price, win_rate, count, ma20, atr_stop])

    if not out:
        print("😴 今日無符合高勝率嘅強勢訊號。")
        return

    # 整理結果
    df_out = pd.DataFrame(out, columns=[
        "Ticker", "策略", "現價", "勝率%", "次數", "MA20", "ATR止蝕"
    ])

    # 格式化數字
    pd.options.display.float_format = '{:.2f}'.format

    # 排序：勝率優先
    df_out = df_out.sort_values(["勝率%", "次數"], ascending=[False, False])
    top3 = df_out.head(5)

    print("\n🏆【今日精選 Top 5】")
    print(top3.to_string(index=False))

    # 輸出檔案
    os.makedirs("soso_reports", exist_ok=True)
    fname = f"soso_reports/daily_scan_{dt.date.today().strftime('%Y%m%d')}.txt"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(f"掃描日期: {dt.date.today()}\n")
        f.write(msg + "\n\n")
        f.write(top3.to_string(index=False))

    print(f"\n📝 報告已儲存: {fname}")
    print("💡 提提你：入市前深呼吸，跟足計劃，嚴守止蝕。")

# 執行
if __name__ == "__main__":
    run()


------------------------------
🤖 蘇蘇投資助理 - 每日掃描
------------------------------

🔍【大市檢查】
🟢 Risk-On（大市健康，適合操作）

🚀 正在掃描 (條件: 勝率≥65% & 次數≥5)... 請稍候...

🏆【今日精選 Top 5】
Ticker      策略     現價   勝率%  次數   MA20  ATR止蝕
   IWM 🔥 VCP形態 285.12 77.63  76 280.16 274.92
   QQQ 🔥 VCP形態 717.54 71.17 222 694.91 694.65
  AMZN 🔥 VCP形態 266.32 65.54 177 267.10 254.08
   JNJ 🔥 VCP形態 234.34 65.15  66 227.18 226.61

📝 報告已儲存: soso_reports/daily_scan_20260524.txt
💡 提提你：入市前深呼吸，跟足計劃，嚴守止蝕。


In [2]:
# -*- coding: utf-8 -*-
"""
蘇蘇指揮官系統 v4.2 (升級版：5年回測 + 顯示次數)
功能：
1. [大市] 三角共振：QQQ + SPY + VIX 聯合判斷。
2. [掃描] 智能分級：只選 A 級強勢股。
3. [數據] 歷史勝率：計算該股過去 5 年在「趨勢向上」時的 20日勝率。
4. [驗證] 回測次數：顯示樣本數量，確認勝率可信度。
5. [風控] 自動止蝕：MA20 / ATR。
"""

import yfinance as yf
import pandas as pd
import numpy as np
import datetime as dt

# --- ⚙️ 蘇蘇個人設定 (請誠實更改) ---
CURRENT_HOLDINGS_COUNT = 0  # 你而家揸緊幾多隻 Swing 股？
MAX_HOLDINGS = 3            # 最多准揸幾多隻？

# --- 監控名單 (Top 100 強勢股 - 擴充版) ---
TOP_100_TICKERS = [
    # --- Mag 7 巨頭 ---
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA",

    # --- 半導體 / AI 硬體 ---
    "AVGO", "AMD", "QCOM", "TXN", "INTC", "IBM", "ORCL", "CRM", "ADBE",
    "MU", "LRCX", "AMAT", "KLAC", "ADI", "NXPI", "MRVL", "ARM", "SMH", "SOXX",

    # --- 軟體 / 雲端 / 大數據 ---
    "NFLX", "PLTR", "NOW", "SNOW", "PANW", "CRWD", "DDOG", "MDB", "NET", "TEAM",

    # --- 金融 / 支付 / 投資 ---
    "JPM", "BAC", "V", "MA", "WFC", "MS", "GS", "BRK-B", "AXP", "BLK", "C", "PYPL",

    # --- 醫療 / 製藥 / 生物科技 ---
    "LLY", "UNH", "JNJ", "ABBV", "MRK", "PFE", "TMO", "AMGN", "ISRG", "VRTX", "REGN", "BMY",

    # --- 消費 / 零售 / 服務 ---
    "COST", "WMT", "HD", "PG", "KO", "PEP", "MCD", "NKE", "SBUX",
    "DIS", "CMG", "MAR", "BKNG", "ABNB", "UBER", "LULU", "TGT", "LOW",

    # --- 工業 / 能源 / 防守 ---
    "CAT", "BA", "GE", "XOM", "CVX", "LIN", "DE", "HON", "UPS", "UNP", "WM", "RSG",

    # --- 加密貨幣相關 ---
    "COIN", "MSTR", "MARA", "RIOT",

    # --- 其他高動能 / 成長股 ---
    "ELF", "CELH", "ANET", "FSLR", "ENPH", "FUTU"
]

# --- 工具函數 ---

def get_data_safe(ticker, period="1y"):
    """安全下載數據"""
    try:
        df = yf.download(ticker, period=period, progress=False, auto_adjust=False)
        if isinstance(df.columns, pd.MultiIndex):
            try: df = df.xs(ticker, level=1, axis=1)
            except: pass
        if 'Adj Close' in df.columns: df['Close'] = df['Adj Close']
        return df
    except: return pd.DataFrame()

def calculate_atr(df, period=14):
    """計算 ATR (波動率)"""
    high = df['High']
    low = df['Low']
    close = df['Close'].shift(1)
    tr = pd.concat([high - low, (high - close).abs(), (low - close).abs()], axis=1).max(axis=1)
    return tr.rolling(period).mean()

def calculate_trend_win_rate(ticker):
    """
    🔥 [升級] 計算趨勢勝率 + 次數
    邏輯：過去 5 年，當股價 > MA20 (趨勢向上) 時買入，持有 20 日後的勝率。
    """
    try:
        # 改成 5年 數據
        df = get_data_safe(ticker, period="5y")
        if df.empty or len(df) < 100: return 0, 0

        df['MA20'] = df['Close'].rolling(20).mean()

        # 條件：當日收市 > MA20 (處於上升趨勢)
        entries = df[df['Close'] > df['MA20']].index

        wins = 0
        total = 0

        for date in entries:
            try:
                idx = df.index.get_loc(date)
                if idx + 20 < len(df): # 確保有 20 日後的數據
                    entry_price = float(df.iloc[idx]['Close'])
                    exit_price = float(df.iloc[idx + 20]['Close'])

                    if exit_price > entry_price:
                        wins += 1
                    total += 1
            except: continue

        win_rate = (wins / total * 100) if total > 0 else 0
        return win_rate, total
    except: return 0, 0

def get_market_resonance():
    """三角共振判斷：QQQ + SPY + VIX"""
    qqq = get_data_safe("QQQ", "1y")
    spy = get_data_safe("SPY", "1y")
    vix = get_data_safe("^VIX", "1y")

    if qqq.empty or spy.empty or vix.empty:
        return "數據不足", "⚠️ 無法判斷", "GRAY", 0

    q_price = qqq['Close'].iloc[-1]
    q_ma50 = qqq['Close'].rolling(50).mean().iloc[-1]
    q_ma200 = qqq['Close'].rolling(200).mean().iloc[-1]

    s_price = spy['Close'].iloc[-1]
    s_ma50 = spy['Close'].rolling(50).mean().iloc[-1]

    v_price = vix['Close'].iloc[-1]

    # 基本牛熊分界 (如果 QQQ 跌穿 MA200，直接紅燈)
    if q_price < q_ma200:
         return "🔴 Risk Off (熊市)", "⛔ 紅燈：停止買入！持有現金/防守。", "RED", v_price

    # 短期共振
    cond_qqq = q_price > q_ma50
    cond_spy = s_price > s_ma50
    cond_vix = v_price < 20

    score = sum([cond_qqq, cond_spy, cond_vix])

    if score == 3:
        return "🟢 全面 Risk On (三角共振)", "✅ 綠燈：積極尋找 A 級強勢股。", "GREEN", v_price
    elif score == 2:
        return "🟡 震盪/分歧 (小心)", "⚠️ 黃燈：只准小注 A 級龍頭，嚴守止蝕。", "YELLOW", v_price
    else:
        return "🔴 Risk Off (轉弱)", "⛔ 紅燈：停止買入！", "RED", v_price

def analyze_stock_grade(df, ticker):
    """智能評級 + 止蝕 + 勝率"""
    close = df['Close']
    price = close.iloc[-1]
    ma20 = close.rolling(20).mean().iloc[-1]
    ma50 = close.rolling(50).mean().iloc[-1]
    ma200 = close.rolling(200).mean().iloc[-1]
    high100 = close.rolling(100).max().iloc[-1]

    # RSI
    delta = close.diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs_val = gain / loss
    rsi = 100 - (100 / (1 + rs_val))
    rsi_curr = rsi.iloc[-1]

    # ATR 止蝕
    atr = calculate_atr(df).iloc[-1]
    stop_atr = price - (atr * 2)

    # 評級邏輯
    grade = "F"
    if price > ma200:
        if price >= 0.95 * high100 and price > ma20 and rsi_curr > 50:
            grade = "A (爆發型)"
        elif price > ma50 and price > ma20:
            grade = "B (趨勢型)"
        elif price > ma20:
            grade = "C (觀察中)"
        else:
            grade = "D (回調)"

    # 只回傳 A 或 B
    if "A" in grade or "B" in grade:
        # 🔥 計算勝率及次數
        win_rate, trade_count = calculate_trend_win_rate(ticker)

        return {
            "代號": ticker,
            "評級": grade,
            "現價": price,
            "歷史勝率": win_rate,
            "回測次數": trade_count,
            "止蝕_MA20": ma20,
            "止蝕_ATR": stop_atr,
            "RSI": rsi_curr
        }
    return None

def run_smart_scan():
    """執行全市場掃描"""
    results = []
    print("🚀 正在掃描並回測 Top 100 (5年數據)... 需時約 60-90 秒...")

    for t in TOP_100_TICKERS:
        try:
            df = get_data_safe(t)
            if df.empty or len(df) < 200: continue

            res = analyze_stock_grade(df, t)
            if res: results.append(res)
        except: continue

    if not results: return pd.DataFrame()

    # 排序：優先睇「評級 A」，然後睇「歷史勝率」
    df = pd.DataFrame(results)
    df = df.sort_values(by=["評級", "歷史勝率"], ascending=[True, False]) # A 排先，勝率高排先
    return df.head(10)

# --- 🚀 蘇蘇指揮官主程式 v4.2 ---

print("="*75)
print(f"👮‍♂️ 蘇蘇指揮官 v4.2 (5年回測 + 次數驗證版) - {dt.datetime.now().strftime('%Y-%m-%d')}")
print("="*75)

# 1. 大市
status, action, color, vix_val = get_market_resonance()

# 2. 額度
overtrade_msg = ""
can_buy = False
if CURRENT_HOLDINGS_COUNT >= MAX_HOLDINGS:
    overtrade_msg = f"⚠️ 持倉爆額 ({CURRENT_HOLDINGS_COUNT}/{MAX_HOLDINGS})！禁止買入，只准賣出。"
else:
    overtrade_msg = f"✅ 額度正常 ({CURRENT_HOLDINGS_COUNT}/{MAX_HOLDINGS})，可尋找機會。"
    can_buy = True

# 3. 掃描
top_picks = pd.DataFrame()
if color != "RED" and can_buy:
    top_picks = run_smart_scan()

# --- 🎯 最終簡報 ---
print("\n【 📌 蘇蘇早晨指令 】\n")

print(f"1️⃣ 大市狀態：{status} (VIX: {vix_val:.2f})")
print(f"   👉 指令：{action}")
print("-" * 65)

print(f"2️⃣ 紀律檢查：{overtrade_msg}")
print("-" * 65)

if not top_picks.empty and can_buy:
    print("3️⃣ 重點出擊 (已過濾勝率 + 次數驗證)：")
    print(f"{'代號':<6} | {'評級':<10} | {'歷史勝率':<8} | {'次數':<5} | {'止蝕 (MA20)':<12} | {'ATR 止蝕'}")
    print("-" * 75)
    for index, row in top_picks.iterrows():
        # 勝率加顏色/標記
        win_str = f"{row['歷史勝率']:.0f}%"
        if row['歷史勝率'] > 60: win_str += "🔥"

        # 次數轉換為整數
        count_val = int(row['回測次數'])

        print(f"{row['代號']:<6} | {row['評級']:<10} | {win_str:<8} | {count_val:<5} | {row['止蝕_MA20']:.2f}         | {row['止蝕_ATR']:.2f}")

    print("\n💡 蘇蘇心法：")
    print("   👉 [勝率 > 60%] + [次數 > 15] = 真強勢股 (穩陣)。")
    print("   👉 [勝率 100%] 但 [次數 < 3] = 樣本太少，要小心。")
    print("   👉 買入即刻 Set 止蝕 (MA20)，然後放低電話，陪老婆囡囡。")

elif not can_buy:
    print("3️⃣ 今日任務：❌ 暫停買入。管理好現有倉位。")
else:
    print("3️⃣ 今日任務：👀 市場無 A 級強勢股。忍手，去健身。")

print("="*75)

👮‍♂️ 蘇蘇指揮官 v4.2 (5年回測 + 次數驗證版) - 2026-05-24
🚀 正在掃描並回測 Top 100 (5年數據)... 需時約 60-90 秒...

【 📌 蘇蘇早晨指令 】

1️⃣ 大市狀態：🟢 全面 Risk On (三角共振) (VIX: 16.70)
   👉 指令：✅ 綠燈：積極尋找 A 級強勢股。
-----------------------------------------------------------------
2️⃣ 紀律檢查：✅ 額度正常 (0/3)，可尋找機會。
-----------------------------------------------------------------
3️⃣ 重點出擊 (已過濾勝率 + 次數驗證)：
代號     | 評級         | 歷史勝率     | 次數    | 止蝕 (MA20)    | ATR 止蝕
---------------------------------------------------------------------------
ARM    | A (爆發型)    | 65%🔥     | 365   | 224.64         | 264.65
SMH    | A (爆發型)    | 62%🔥     | 749   | 542.16         | 534.69
LIN    | A (爆發型)    | 61%🔥     | 710   | 505.62         | 498.23
CRWD   | A (爆發型)    | 60%🔥     | 698   | 536.66         | 609.01
AAPL   | A (爆發型)    | 60%      | 706   | 289.22         | 297.30
SOXX   | A (爆發型)    | 60%      | 712   | 496.31         | 492.97
LRCX   | A (爆發型)    | 59%      | 699   | 280.09         | 274.73
LLY    | A (爆發型)    | 59%      | 722   | 972.75   

In [3]:
# -*- coding: utf-8 -*-

"""

蘇蘇VCP (嚴選版：勝率 > 60% + 止蝕位)

-----------------------------------------------------

核心規則：

1. [嚴格篩選]：只顯示歷史勝率 >= 60% 的股票，低勝率直接隱藏。

2. [風控顯示]：明確列出 MA20 及 ATR 止蝕價位。

3. [策略]：包含 VCP 形態及急跌反彈策略。

"""



import yfinance as yf

import pandas as pd

import numpy as np

import datetime



# ======== 1. 目標股票 (Top 50 + 強勢股) ========

US_LARGE_CAP = [

    "AAPL", "MSFT", "AMZN", "GOOG", "GOOGL", "META", "NVDA", "TSLA",

    "BRK-B", "JPM", "V", "MA", "JNJ", "PG", "HD", "AVGO",

    "XOM", "LLY", "UNH", "COST", "WMT", "BAC", "ADBE",

    "ORCL", "KO", "PFE", "CSCO", "MRK", "CRM", "PEP",

    "TMO", "ABT", "NFLX", "DIS", "MCD", "AMD", "TXN",

    "INTC", "HON", "IBM", "LIN", "QCOM", "AMAT", "CAT",

    "LOW", "GE", "UPS", "CVX", "SPY", "QQQ", "COIN", "MSTR", "PLTR",

    "ISRG", "MAR", "SMH", "ARM", "CRWD", "PANW", "ELF", "CELH"

]



# ======== 2. 大市濾網 ========

def check_market_regime():

    print("🔍 正在檢查大市趨勢 (QQQ vs MA200)...")

    try:

        qqq = yf.download("QQQ", period="1y", progress=False, auto_adjust=False)

        if isinstance(qqq.columns, pd.MultiIndex):

            try: qqq = qqq.xs("QQQ", level=1, axis=1)

            except: pass

        if 'Adj Close' in qqq.columns: qqq['Close'] = qqq['Adj Close']



        qqq["MA200"] = qqq["Close"].rolling(200).mean()

        current_price = qqq["Close"].iloc[-1]

        ma200 = qqq["MA200"].iloc[-1]



        print(f"   👉 QQQ 現價: {current_price:.2f} | MA200: {ma200:.2f}")



        if current_price > ma200:

            print("   ✅ 大市強勢 (Bull Market)，允許掃描。\n")

            return True

        else:

            print("\n🛑 熊市警告：QQQ < MA200，系統已自動鎖定，禁止買入。")

            return False

    except:

        return False



# ======== 3. 指標計算 ========

def calc_rsi(series, period=14):

    delta = series.diff()

    up = delta.clip(lower=0)

    down = -delta.clip(upper=0)

    rsi = 100 - 100 / (1 + up.rolling(period).mean() / (down.rolling(period).mean() + 1e-10))

    return rsi



def calc_atr(df, period=14):

    high, low, close = df['High'], df['Low'], df['Close']

    tr = pd.concat([high - low, (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)

    return tr.rolling(period).mean()



# ======== 4. 策略邏輯 ========

def strategy_vcp_pivot(df):

    """VCP 買入點偵測"""

    close = df["Close"]

    volume = df["Volume"]

    ma50 = close.rolling(50).mean()

    ma150 = close.rolling(150).mean()

    ma200 = close.rolling(200).mean()

    high_52w = close.rolling(250).max()

    low_52w = close.rolling(250).min()



    # 趨勢條件

    trend_cond = (close > ma50) & (ma50 > ma150) & (ma150 > ma200)

    trend_cond &= (close > low_52w * 1.3)

    trend_cond &= (close > high_52w * 0.75)



    # 波動收縮

    std_10 = close.rolling(10).std()

    std_50 = close.rolling(50).std()

    contraction = std_10 < (std_50 * 0.6)



    # 量縮

    vol_ma50 = volume.rolling(50).mean()

    vol_dry = volume.rolling(5).mean() < vol_ma50



    return trend_cond & contraction & vol_dry



def strategy_drop_ma20(df):

    if "MA20" not in df.columns: df["MA20"] = df["Close"].rolling(20).mean()

    drop_cond = (df["Close"] / df["Close"].shift(1) - 1) < -0.04

    recent_drop = drop_cond.rolling(3).max().fillna(0).astype(bool)

    cross_ma20 = (df["Close"] > df["MA20"]) & (df["Close"].shift(1) <= df["MA20"])

    return recent_drop & cross_ma20



# ======== 5. 回測功能 (10% 止蝕) ========

def backtest_fixed_10pct(df, signal_series, periods=[10, 20, 60]):

    results = {}

    valid_signals = signal_series[signal_series]

    entry_indices = valid_signals.index



    for days in periods:

        if len(entry_indices) == 0:

            results[days] = 0

            continue



        wins = 0

        total = 0

        testable_entries = entry_indices[entry_indices < df.index[-days]]



        for date in testable_entries:

            try:

                idx = df.index.get_loc(date)

                entry_price = df.iloc[idx]["Close"]

                stop_loss_price = entry_price * 0.90



                period_end = min(idx + days + 1, len(df))

                holding_period = df.iloc[idx+1 : period_end]

                min_price = holding_period["Low"].min()



                if min_price < stop_loss_price:

                    total += 1

                    continue



                exit_price = df.iloc[idx+days]["Close"]

                if exit_price > entry_price:

                    wins += 1

                total += 1

            except: continue

        win_rate = (wins / total * 100) if total > 0 else 0

        results[days] = win_rate



    return results, len(entry_indices)



# ======== 6. 主程序 ========

def scan_market():

    print(f"🚀 啟動蘇蘇全能掃描系統 v5.1 (嚴選版) - {datetime.date.today()}\n")



    if not check_market_regime(): return



    print("⏳ 正在掃描... 只顯示勝率 > 60% 之強勢股... 需時約 60 秒...")

    final_list = []



    for ticker in US_LARGE_CAP:

        try:

            df = yf.download(ticker, period="2y", progress=False, auto_adjust=False)

            if isinstance(df.columns, pd.MultiIndex):

                try: df = df.xs(ticker, level=1, axis=1)

                except: pass

            if 'Adj Close' in df.columns: df['Close'] = df['Adj Close']



            if df.empty or len(df) < 200: continue



            # 準備指標

            if "MA20" not in df.columns: df["MA20"] = df["Close"].rolling(20).mean()

            if "ATR" not in df.columns: df["ATR"] = calc_atr(df)



            strategies = [

                ("🔥VCP突破", strategy_vcp_pivot(df)),

                ("急跌+MA20", strategy_drop_ma20(df))

            ]



            for strat_name, strat_series in strategies:

                if strat_series.iloc[-1]:

                    win_rates, count = backtest_fixed_10pct(df, strat_series)



                    # 基本過濾：次數太少唔計

                    min_count = 1 if "VCP" in strat_name else 3



                    if count >= min_count:

                        # 💥 這裡進行嚴格過濾：勝率 < 60% 唔要

                        if win_rates[20] < 60:

                            continue



                        current_ma20 = df["MA20"].iloc[-1]

                        current_atr = df["ATR"].iloc[-1]

                        current_price = df["Close"].iloc[-1]

                        stop_atr = current_price - (current_atr * 2.0)



                        final_list.append({

                            "代號": ticker,

                            "策略": strat_name,

                            "現價": current_price,

                            "Win20": win_rates[20],

                            "Win60": win_rates[60],

                            "次數": count,

                            "StopMA20": current_ma20,

                            "StopATR": stop_atr

                        })

        except Exception as e:

            continue



    # --- 顯示結果 ---

    print("\n" + "="*110)

    print(f"📊 蘇蘇嚴選結果 (勝率 > 60% 保證)")

    print("="*110)



    if not final_list:

        print("今日無符合「勝率 > 60%」條件的股票。市場可能多數係弱勢反彈，建議忍手。")

    else:

        # VCP 排先，然後按勝率排

        final_list.sort(key=lambda x: (0 if "VCP" in x["策略"] else 1, -x["Win20"]))



        print(f"{'代號':<6} {'策略名稱':<10} {'現價':<8} {'20日勝率':<8} | {'MA20 止蝕':<10} {'ATR 止蝕':<10} | {'蘇蘇建議'}")

        print("-" * 110)



        for item in final_list:

            w20 = item['Win20']

            advice = ""



            if "VCP" in item["策略"]:

                advice = "💎潛在爆發"

            elif w20 >= 75: advice = "🔥重注 (極穩)"

            elif w20 >= 60: advice = "✅中注 (值博)"



            # 顯示

            print(f"{item['代號']:<6} {item['策略']:<10} {item['現價']:.2f}     {item['Win20']:.0f}%       | {item['StopMA20']:.2f}       {item['StopATR']:.2f}       | {advice}")



    print("-" * 110)

    print("💡 操作指引：")

    print("   👉 既然勝率有 60% 以上，買入後請立即在證券 App 設定 [MA20 止蝕] 價位。")

    print("   👉 如果跌穿，無需猶豫，立即執行紀律！")



if __name__ == "__main__":

    scan_market()

🚀 啟動蘇蘇全能掃描系統 v5.1 (嚴選版) - 2026-05-24

🔍 正在檢查大市趨勢 (QQQ vs MA200)...
   👉 QQQ 現價: 717.54 | MA200: 613.60
   ✅ 大市強勢 (Bull Market)，允許掃描。

⏳ 正在掃描... 只顯示勝率 > 60% 之強勢股... 需時約 60 秒...

📊 蘇蘇嚴選結果 (勝率 > 60% 保證)
代號     策略名稱       現價       20日勝率    | MA20 止蝕    ATR 止蝕     | 蘇蘇建議
--------------------------------------------------------------------------------------------------------------
AVGO   🔥VCP突破     414.14     89%       | 419.09       381.21       | 💎潛在爆發
MAR    🔥VCP突破     369.15     89%       | 356.50       351.77       | 💎潛在爆發
GOOG   🔥VCP突破     379.38     71%       | 382.29       360.29       | 💎潛在爆發
QQQ    🔥VCP突破     717.54     70%       | 694.91       694.65       | 💎潛在爆發
JNJ    🔥VCP突破     234.34     65%       | 227.18       226.61       | 💎潛在爆發
--------------------------------------------------------------------------------------------------------------
💡 操作指引：
   👉 既然勝率有 60% 以上，買入後請立即在證券 App 設定 [MA20 止蝕] 價位。
   👉 如果跌穿，無需猶豫，立即執行紀律！


In [4]:
# -*- coding: utf-8 -*-
"""
蘇蘇全能掃描系統 v4.6 (5年回測 + 顯示次數版)
-----------------------------------------------------
核心邏輯：
1. [回測標準]：統一用「固定 10% 止蝕」計算歷史勝率。
   👉 測試隻股有幾「穩陣」，如果經常 Chok 穿 10% 就算輸。
2. [實戰建議]：提供「MA20」及「ATR」動態止蝕位。
   👉 讓你跟隨趨勢操作，避免過早離場。
3. [顯示更新]：回測範圍改為 5年，並新增「次數」顯示，提升數據可信度。
"""

import yfinance as yf
import pandas as pd
import numpy as np
import datetime

# ======== 1. 目標股票 ========
US_LARGE_CAP = [
    "AAPL", "MSFT", "AMZN", "GOOG", "GOOGL", "META", "NVDA", "TSLA",
    "BRK-B", "JPM", "V", "MA", "JNJ", "PG", "HD", "AVGO",
    "XOM", "LLY", "UNH", "COST", "WMT", "BAC", "ADBE",
    "ORCL", "KO", "PFE", "CSCO", "MRK", "CRM", "PEP",
    "TMO", "ABT", "NFLX", "DIS", "MCD", "AMD", "TXN",
    "INTC", "HON", "IBM", "LIN", "QCOM", "AMAT", "CAT",
    "LOW", "GE", "UPS", "CVX", "SPY", "QQQ", "COIN", "MSTR", "PLTR",
    "ISRG", "MAR", "SMH", "TQQQ", "SOXL"
]

# ======== 2. 大市濾網 ========
def check_market_regime():
    print("🔍 正在檢查大市趨勢 (QQQ vs MA200)...")
    try:
        qqq = yf.download("QQQ", period="1y", progress=False, auto_adjust=False)
        if isinstance(qqq.columns, pd.MultiIndex):
            try: qqq = qqq.xs("QQQ", level=1, axis=1)
            except: pass
        if 'Adj Close' in qqq.columns: qqq['Close'] = qqq['Adj Close']

        qqq["MA200"] = qqq["Close"].rolling(200).mean()
        current_price = float(qqq["Close"].iloc[-1])
        ma200 = float(qqq["MA200"].iloc[-1])

        print(f"   👉 QQQ 現價: {current_price:.2f} | MA200: {ma200:.2f}")

        if current_price > ma200:
            print("   ✅ 大市強勢 (Bull Market)，允許掃描。\n")
            return True
        else:
            print("\n🛑 熊市警告：QQQ < MA200，系統已自動鎖定，禁止買入。")
            return False
    except Exception as e:
        print(f"Check market error: {e}")
        return False

# ======== 3. 指標計算 ========
def calc_rsi(series, period=14):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    rsi = 100 - 100 / (1 + up.rolling(period).mean() / (down.rolling(period).mean() + 1e-10))
    return rsi

def calc_atr(df, period=14):
    high, low, close = df['High'], df['Low'], df['Close']
    tr = pd.concat([high - low, (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ======== 4. 策略邏輯 ========
def strategy_rsi_reversal(df):
    df["RSI"] = calc_rsi(df["Close"])
    return (df["RSI"].shift(1) < 30) & (df["Close"] > df["Close"].shift(1))

def strategy_drop_ma20(df):
    if "MA20" not in df.columns: df["MA20"] = df["Close"].rolling(20).mean()
    drop_cond = (df["Close"] / df["Close"].shift(1) - 1) < -0.04
    recent_drop = drop_cond.rolling(3).max().fillna(0).astype(bool)
    cross_ma20 = (df["Close"] > df["MA20"]) & (df["Close"].shift(1) <= df["MA20"])
    return recent_drop & cross_ma20

def strategy_3bar_reversal(df):
    down2 = (df["Close"] < df["Close"].shift(1)) & (df["Close"].shift(1) < df["Close"].shift(2))
    strong_reversal = df["Close"] > df["High"].shift(1)
    return down2.shift(1) & strong_reversal

def strategy_atr_oversold(df):
    if "ATR" not in df.columns: df["ATR"] = calc_atr(df)
    panic_drop = (df["High"].shift(1) - df["Low"].shift(1)) > (df["ATR"].shift(1) * 2.0)
    rebound = df["Close"] > df["High"].shift(1)
    return panic_drop & rebound

# ======== 5. 🔥 多週期回測 (固定 10% 止蝕) ========
def backtest_fixed_10pct(df, signal_series, periods=[10, 20, 60]):
    """
    計算勝率：假設買入後，若任何時間跌穿 10% 即止蝕 (當輸)。
    """
    results = {}
    valid_signals = signal_series[signal_series]
    entry_indices = valid_signals.index

    for days in periods:
        if len(entry_indices) == 0:
            results[days] = 0
            continue

        wins = 0
        total = 0
        testable_entries = entry_indices[entry_indices < df.index[-days]]

        for date in testable_entries:
            try:
                idx = df.index.get_loc(date)
                entry_price = float(df.iloc[idx]["Close"])
                stop_loss_price = entry_price * 0.90 # 固定 10% 止蝕

                period_end = min(idx + days + 1, len(df))
                holding_period = df.iloc[idx+1 : period_end]

                # 檢查持有期間有冇觸發 10% 止蝕
                min_price = holding_period["Low"].min()

                if min_price < stop_loss_price:
                    # 觸發止蝕，當輸
                    total += 1
                    continue

                # 如果無止蝕，檢查最終收市價
                exit_price = float(df.iloc[idx+days]["Close"])
                if exit_price > entry_price:
                    wins += 1

                total += 1
            except: continue

        win_rate = (wins / total * 100) if total > 0 else 0
        results[days] = win_rate

    return results, len(entry_indices)

# ======== 6. 主程序 ========
def scan_market():
    print(f"🚀 啟動蘇蘇全能掃描系統 v4.6 (5年數據 + 顯示次數) - {datetime.date.today()}\n")

    if not check_market_regime(): return

    print("⏳ 正在掃描... 回測標準：5年數據 + 固定 10% 止蝕... 需時約 60 秒...")
    final_list = []

    for ticker in US_LARGE_CAP:
        try:
            # 改為 5年 數據
            df = yf.download(ticker, period="5y", progress=False, auto_adjust=False)
            if isinstance(df.columns, pd.MultiIndex):
                try: df = df.xs(ticker, level=1, axis=1)
                except: pass
            if 'Adj Close' in df.columns: df['Close'] = df['Adj Close']

            if df.empty or len(df) < 100: continue

            # 準備指標
            if "MA20" not in df.columns: df["MA20"] = df["Close"].rolling(20).mean()
            if "ATR" not in df.columns: df["ATR"] = calc_atr(df)

            strategies = [
                ("RSI反彈", strategy_rsi_reversal(df)),
                ("急跌+MA20", strategy_drop_ma20(df)),
                ("3日反轉", strategy_3bar_reversal(df)),
                ("ATR恐慌", strategy_atr_oversold(df))
            ]

            for strat_name, strat_series in strategies:
                if strat_series.iloc[-1]:
                    # 🔥 使用 10% 固定止蝕進行回測
                    win_rates, count = backtest_fixed_10pct(df, strat_series)

                    if count >= 3:
                        # 準備「實戰建議」的動態止蝕位
                        current_ma20 = float(df["MA20"].iloc[-1])
                        current_atr = float(df["ATR"].iloc[-1])
                        current_price = float(df["Close"].iloc[-1])

                        # ATR x 2 止蝕位
                        stop_atr = current_price - (current_atr * 2.0)

                        final_list.append({
                            "代號": ticker,
                            "策略": strat_name,
                            "現價": current_price,
                            "Win20": win_rates[20],
                            "Win60": win_rates[60],
                            "次數": count,
                            "StopMA20": current_ma20,
                            "StopATR": stop_atr
                        })
        except Exception as e:
            continue

    # --- 顯示結果 ---
    print("\n" + "="*125)
    print(f"📊 蘇蘇智能選股結果")
    print("="*125)

    if not final_list:
        print("今日無符合條件的股票。")
    else:
        # 按 20日勝率 排序
        final_list.sort(key=lambda x: x["Win20"], reverse=True)

        # 標題欄加入 Win60 和 次數
        print(f"{'代號':<6} {'策略名稱':<10} {'現價':<8} {'Win20':<8} {'Win60':<8} {'次數':<6} | {'MA20 止蝕':<10} {'ATR 止蝕':<10} | {'蘇蘇建議'}")
        print("-" * 125)

        for item in final_list:
            w20 = item['Win20']
            w60 = item['Win60']
            count = item['次數']

            advice = ""
            # 綜合考慮短線同長線
            if w20 >= 70 and w60 >= 50 and count >= 5: advice = "🔥重注"
            elif w20 >= 60 and count >= 5: advice = "✅中注"
            elif w20 >= 50: advice = "👀小注"
            elif w20 < 40: advice = "⚠️極危"
            else: advice = "⚠️小心"

            # 顯示內容加入 次數
            print(f"{item['代號']:<6} {item['策略']:<10} {item['現價']:.2f}     {w20:<8.0f} {w60:<8.0f} {count:<6} | {item['StopMA20']:.2f}       {item['StopATR']:.2f}       | {advice}")

    print("-" * 125)
    print("💡 蘇蘇專屬指引：")
    print("   1. [次數]：過去5年出現過幾多次訊號。次數多且勝率高，信賴度UP！")
    print("   2. [Win60]：60日生存率，睇長線趨勢係咪真係穩。")
    print("   3. [實戰]：買入後請用「MA20」或「ATR」做移動止蝕，讓利潤奔跑。")

if __name__ == "__main__":
    scan_market()

🚀 啟動蘇蘇全能掃描系統 v4.6 (5年數據 + 顯示次數) - 2026-05-24

🔍 正在檢查大市趨勢 (QQQ vs MA200)...
   👉 QQQ 現價: 717.54 | MA200: 613.60
   ✅ 大市強勢 (Bull Market)，允許掃描。

⏳ 正在掃描... 回測標準：5年數據 + 固定 10% 止蝕... 需時約 60 秒...

📊 蘇蘇智能選股結果
代號     策略名稱       現價       Win20    Win60    次數     | MA20 止蝕    ATR 止蝕     | 蘇蘇建議
-----------------------------------------------------------------------------------------------------------------------------
PEP    RSI反彈      150.57     67       58       65     | 152.73       144.38       | ✅中注
PEP    3日反轉       150.57     50       0        3      | 152.73       144.38       | 👀小注
UNH    3日反轉       388.47     33       0        4      | 379.63       369.72       | ⚠️極危
-----------------------------------------------------------------------------------------------------------------------------
💡 蘇蘇專屬指引：
   1. [次數]：過去5年出現過幾多次訊號。次數多且勝率高，信賴度UP！
   2. [Win60]：60日生存率，睇長線趨勢係咪真係穩。
   3. [實戰]：買入後請用「MA20」或「ATR」做移動止蝕，讓利潤奔跑。


In [5]:
# ==========================================
# 1. 手動輸入 S&P 100 名單
# ==========================================
SP100_TICKERS = [
    "AAPL", "ABBV", "ABT", "ACN", "ADBE", "AIG", "AMD", "AMGN", "AMT", "AMZN",
    "AXP", "BA", "BAC", "BK", "BKNG", "BLK", "BMY", "BRK-B", "C", "CAT",
    "CHTR", "CL", "CMCSA", "COF", "COP", "COST", "CRM", "CSCO", "CVS", "CVX",
    "DE", "DHR", "DIS", "DOW", "DUK", "EMR", "EXC", "F", "FDX", "GD",
    "GE", "GILD", "GM", "GOOG", "GOOGL", "GS", "HD", "HON", "IBM", "INTC",
    "JNJ", "JPM", "KHC", "KO", "LIN", "LLY", "LMT", "LOW", "MA", "MCD",
    "MDLZ", "MDT", "MET", "META", "MMM", "MO", "MRK", "MS", "MSFT", "NEE",
    "NFLX", "NKE", "NVDA", "ORCL", "PEP", "PFE", "PG", "PM", "PYPL", "QCOM",
    "RTX", "SBUX", "SCHW", "SO", "SPG", "T", "TGT", "TMO", "TMUS", "TSLA",
    "TXN", "UNH", "UNP", "UPS", "USB", "V", "VZ", "WFC", "WMT", "XOM"
]

SECTOR_ETFS = {
    'XLK': '科技', 'XLE': '能源', 'XLF': '金融', 'XLV': '醫療',
    'XLY': '非必需消費', 'XLP': '必需消費', 'XLI': '工業', 'XLC': '通訊',
    'XLU': '公用/電力', 'XLB': '原材料', 'XME': '金屬礦業', 'SMH': '半導體',
    'SPY': '標普500基準'
}

# ==========================================
# 2. 定義功能函數 (修復了 yfinance 下載問題)
# ==========================================

def get_sector_performance():
    """分析板塊強弱"""
    print(f"\n🔄 正在分析板塊資金流向...")
    tickers = list(SECTOR_ETFS.keys())

    # 強制 auto_adjust=False 以獲取原始數據結構
    try:
        data = yf.download(tickers, period="2mo", progress=False, auto_adjust=False)
    except Exception as e:
        print(f"板塊數據下載錯誤: {e}")
        return pd.DataFrame()

    # 兼容檢查：搵唔搵到 Adj Close，搵唔到就用 Close
    price_col = 'Adj Close' if 'Adj Close' in data else 'Close'

    performance = []
    for ticker in tickers:
        try:
            if ticker not in data[price_col].columns: continue

            prices = data[price_col][ticker].dropna()
            if len(prices) < 20: continue

            curr = prices.iloc[-1]
            w1 = prices.iloc[-5]
            m1 = prices.iloc[-20]

            pct_w = ((curr - w1) / w1) * 100
            pct_m = ((curr - m1) / m1) * 100

            performance.append({
                '代號': ticker,
                '板塊': SECTOR_ETFS[ticker],
                '1週升跌%': round(pct_w, 2),
                '1月升跌%': round(pct_m, 2)
            })
        except:
            continue

    df = pd.DataFrame(performance)
    if not df.empty:
        df = df.sort_values(by='1週升跌%', ascending=False).reset_index(drop=True)
    return df

def calculate_atr(df, period=14):
    """計算 ATR"""
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = ranges.max(axis=1)
    return true_range.rolling(window=period).mean()

def screen_stocks(tickers):
    """
    S&P 100 篩選器
    """
    print(f"\n🔍 正在篩選 {len(tickers)} 隻 S&P 100 成分股 (請稍等)...")

    tickers_with_spy = tickers + ['SPY']

    try:
        # 強制 auto_adjust=False
        data = yf.download(tickers_with_spy, period="6mo", progress=False, auto_adjust=False)
    except Exception as e:
        print(f"個股數據下載錯誤: {e}")
        return pd.DataFrame()

    price_col = 'Adj Close' if 'Adj Close' in data else 'Close'

    results = []

    # 獲取 SPY 表現作為基準
    try:
        spy_prices = data[price_col]['SPY'].dropna()
        spy_ret_3m = (spy_prices.iloc[-1] - spy_prices.iloc[-60]) / spy_prices.iloc[-60]
    except:
        spy_ret_3m = 0

    for ticker in tickers:
        try:
            # 檢查數據是否存在
            if ticker not in data['Close'].columns: continue

            # 建立單一股票 DataFrame
            df = pd.DataFrame({
                'High': data['High'][ticker],
                'Low': data['Low'][ticker],
                'Close': data['Close'][ticker],
                'Adj Close': data[price_col][ticker],
                'Volume': data['Volume'][ticker]
            }).dropna()

            if len(df) < 60: continue

            curr_price = df['Adj Close'].iloc[-1]

            # 條件 1: 趨勢 (大於 50SMA)
            sma50 = df['Adj Close'].rolling(50).mean().iloc[-1]
            trend_ok = curr_price > sma50

            # 條件 2: 波動率 (ATR) - 稍微放寬至 2.0% 因為是大價股
            atr = calculate_atr(df).iloc[-1]
            atr_pct = (atr / curr_price) * 100
            vol_ok = atr_pct > 2.0

            # 條件 3: 相對強度 (跑贏 SPY)
            stock_ret_3m = (curr_price - df['Adj Close'].iloc[-60]) / df['Adj Close'].iloc[-60]
            rs_ok = stock_ret_3m > spy_ret_3m

            # 條件 4: 成交額 (> 5000萬美金)
            avg_vol = df['Volume'].rolling(20).mean().iloc[-1]
            liq_ok = (avg_vol * curr_price) > 50_000_000

            score = sum([trend_ok, vol_ok, rs_ok, liq_ok])

            # 只要有 3 分就顯示
            if score >= 3:
                results.append({
                    'Ticker': ticker,
                    '現價': round(curr_price, 2),
                    '大於50SMA': '✅' if trend_ok else '❌',
                    '波動(ATR)%': round(atr_pct, 2),
                    '高波動': '✅' if vol_ok else '❌',
                    '跑贏大市': '✅' if rs_ok else '❌',
                    '總分': score
                })
        except:
            continue

    df_res = pd.DataFrame(results)
    if not df_res.empty:
        # 按波動率排序 (Trader 喜歡波動)
        return df_res.sort_values(by='波動(ATR)%', ascending=False)
    else:
        return pd.DataFrame()

# ==========================================
# 3. 執行程式
# ==========================================

if __name__ == "__main__":
    # A. 板塊分析
    print("--- 📊 步驟一：板塊熱度 ---")
    sector_df = get_sector_performance()
    if not sector_df.empty:
        print(sector_df.to_string(index=False))

    print("\n" + "="*50 + "\n")

    # B. 個股篩選
    print("--- 🕵️ 步驟二：S&P 100 強勢股篩選 ---")
    final_df = screen_stocks(SP100_TICKERS)

    if not final_df.empty:
        print("\n🔥 符合條件的強勢股 (按波動率排序)：")
        print(final_df.to_string(index=False))

        # 簡單提取前 3 名建議
        top3 = final_df.head(3)['Ticker'].tolist()
        print(f"\n💡 建議下一步：將 {', '.join(top3)} 放入 AI 進行新聞與基本面分析。")
    else:
        print("沒有股票符合篩選條件。")

--- 📊 步驟一：板塊熱度 ---

🔄 正在分析板塊資金流向...
 代號      板塊  1週升跌%  1月升跌%
SMH     半導體   5.52  13.84
XLK      科技   3.46  12.34
XME    金屬礦業   3.41  -2.60
XLU   公用/電力   3.21  -1.82
XLV      醫療   2.86   4.48
XLY   非必需消費   2.46   1.14
SPY 標普500基準   0.95   4.26
XLI      工業   0.60  -0.43
XLF      金融   0.39   0.25
XLB     原材料   0.14  -2.88
XLP    必需消費  -1.28   2.99
XLC      通訊  -1.30  -0.30
XLE      能源  -1.80   4.79


--- 🕵️ 步驟二：S&P 100 強勢股篩選 ---

🔍 正在篩選 100 隻 S&P 100 成分股 (請稍等)...

🔥 符合條件的強勢股 (按波動率排序)：
Ticker      現價 大於50SMA  波動(ATR)% 高波動 跑贏大市  總分
  INTC  119.84       ✅      8.48   ✅    ✅   4
  QCOM  238.16       ✅      8.36   ✅    ✅   4
   AMD  467.51       ✅      6.63   ✅    ✅   4
     F   14.93       ✅      4.90   ✅    ❌   3
  ORCL  192.08       ✅      4.60   ✅    ✅   4
  TSLA  426.01       ✅      4.27   ✅    ❌   3
   DOW   36.01       ❌      4.03   ✅    ✅   3
   TGT  125.60       ✅      3.93   ✅    ✅   4
  NVDA  215.33       ✅      3.85   ✅    ✅   4
  CSCO  120.41       ✅      3.60   ✅    ✅   4
   EMR

In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime

# ==========================================
# 1. 設定目標清單 (ETF + Crypto)
# ==========================================
# Yahoo Finance 格式: Crypto 必須加 "-USD"
target_tickers = ['VOO', 'QQQ', 'ETH-USD', 'BNB-USD', 'BTC-USD']

# ==========================================
# 2. 核心算法：計算 TD 9轉 序列 & 回測勝率
# ==========================================
def analyze_stock(ticker, df):
    # 確保數據足夠
    if len(df) < 80: return None # 提高數據要求以配合60日回測

    close = df['Close']
    lag4 = close.shift(4)

    # --- A. 計算每日的 Setup 數值 (Vectorized) ---
    buy_setup = np.zeros(len(close), dtype=int)
    sell_setup = np.zeros(len(close), dtype=int)

    c_vals = close.values
    l4_vals = lag4.values

    curr_buy = 0
    curr_sell = 0

    for i in range(4, len(c_vals)):
        # Low 9 (潛在買入)
        if c_vals[i] < l4_vals[i]:
            curr_buy += 1
        else:
            curr_buy = 0
        buy_setup[i] = curr_buy

        # High 9 (潛在賣出)
        if c_vals[i] > l4_vals[i]:
            curr_sell += 1
        else:
            curr_sell = 0
        sell_setup[i] = curr_sell

    # --- B. 檢查「今日」最新狀態 ---
    last_buy = buy_setup[-1]
    last_sell = sell_setup[-1]

    current_signal = "無"
    if last_buy >= 9:
        current_signal = f"🔴低{last_buy}"
    elif last_buy >= 1:
        current_signal = f"跌:{last_buy}"
    elif last_sell >= 9:
        current_signal = f"🟢高{last_sell}"
    elif last_sell >= 1:
        current_signal = f"升:{last_sell}"

    # --- C. 多時段歷史回測 (10, 30, 60日) ---
    # 儲存不同天數的勝負結果
    wins_buy = {10: [], 30: [], 60: []}
    wins_sell = {10: [], 30: [], 60: []}

    # 1. 回測買入訊號 (低 9)
    buy_indices = np.where(buy_setup[:-1] == 9)[0]
    for idx in buy_indices:
        entry = c_vals[idx]
        for d in [10, 30, 60]:
            if idx + d < len(c_vals):
                exit_price = c_vals[idx + d]
                # 買入後升 = Win
                wins_buy[d].append(1 if exit_price > entry else 0)

    # 2. 回測賣出訊號 (高 9)
    sell_indices = np.where(sell_setup[:-1] == 9)[0]
    for idx in sell_indices:
        entry = c_vals[idx]
        for d in [10, 30, 60]:
            if idx + d < len(c_vals):
                exit_price = c_vals[idx + d]
                # 賣出後跌 = Win
                wins_sell[d].append(1 if exit_price < entry else 0)

    # 計算勝率 Helper
    def calc(lst):
        return (sum(lst) / len(lst) * 100) if lst else 0

    return {
        "Ticker": ticker.replace("-USD", ""),
        "Price": round(c_vals[-1], 2),
        "Signal": current_signal,

        "Buy_Count": len(buy_indices),
        "Buy_10": calc(wins_buy[10]),
        "Buy_30": calc(wins_buy[30]),
        "Buy_60": calc(wins_buy[60]),

        "Sell_Count": len(sell_indices),
        "Sell_10": calc(wins_sell[10]),
        "Sell_30": calc(wins_sell[30]),
        "Sell_60": calc(wins_sell[60]),
    }

# ==========================================
# 3. 主程序
# ==========================================
print(f"正在下載 {target_tickers} 數據並進行多時段回測...\n")

data = yf.download(target_tickers, period="2y", interval="1d", group_by='ticker', progress=False)

# 設定顯示格式
header = f"{'資產':<8} {'現價':<8} {'訊號':<8} | {'低9買入勝率 (10/30/60日)':<28} | {'高9賣出勝率 (10/30/60日)':<28}"
print("-" * 95)
print(header)
print("-" * 95)

for ticker in target_tickers:
    try:
        # 處理數據結構
        if ticker in data.columns or (isinstance(data.columns, pd.MultiIndex) and ticker in data.columns.levels[0]):
             if isinstance(data.columns, pd.MultiIndex):
                 df = data[ticker].dropna()
             else:
                 df = data.dropna()
        else:
             continue

        if df.empty: continue

        r = analyze_stock(ticker, df)

        if r:
            # 格式化勝率字串
            b_stats = f"N={r['Buy_Count']:<2} {r['Buy_10']:.0f}%  {r['Buy_30']:.0f}%  {r['Buy_60']:.0f}%"
            s_stats = f"N={r['Sell_Count']:<2} {r['Sell_10']:.0f}%  {r['Sell_30']:.0f}%  {r['Sell_60']:.0f}%"

            # 加火把 🔥 (如果 30日勝率 > 65%)
            if r['Buy_30'] > 65: b_stats += "🔥"
            if r['Sell_30'] > 65: s_stats += "🔥"

            print(f"{r['Ticker']:<8} {r['Price']:<8} {r['Signal']:<8} | {b_stats:<28} | {s_stats:<28}")

    except Exception as e:
        pass

print("-" * 95)
print("註: N=出現次數 | 百分比代表持有該日數後的獲利機率 | 🔥=30日勝率>65%")

正在下載 ['VOO', 'QQQ', 'ETH-USD', 'BNB-USD', 'BTC-USD'] 數據並進行多時段回測...

-----------------------------------------------------------------------------------------------
資產       現價       訊號       | 低9買入勝率 (10/30/60日)           | 高9賣出勝率 (10/30/60日)          
-----------------------------------------------------------------------------------------------
VOO      685.55   升:2      | N=2  0%  50%  100%           | N=17 47%  13%  13%          
QQQ      717.54   升:2      | N=2  50%  50%  100%          | N=13 46%  50%  42%          
ETH      2095.02  跌:1      | N=6  60%  20%  40%           | N=9  44%  56%  62%          
BNB      655.43   升:4      | N=6  33%  67%  50%🔥          | N=11 45%  30%  30%          
BTC      76375.17 跌:3      | N=9  67%  33%  56%           | N=9  56%  56%  56%          
-----------------------------------------------------------------------------------------------
註: N=出現次數 | 百分比代表持有該日數後的獲利機率 | 🔥=30日勝率>65%


In [7]:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime

# ==========================================
# 1. 設定目標清單 (ETF + Crypto)
# ==========================================
# Yahoo Finance 格式: Crypto 必須加 "-USD"
target_tickers = ['VOO', 'QQQ', 'ETH-USD', 'BNB-USD', 'BTC-USD']

# ==========================================
# 2. 核心算法：計算 TD 9轉 序列 & 回測勝率
# ==========================================
def analyze_stock(ticker, df):
    # 確保數據足夠
    if len(df) < 80: return None # 提高數據要求以配合60日回測

    close = df['Close']
    lag4 = close.shift(4)

    # --- A. 計算每日的 Setup 數值 (Vectorized) ---
    buy_setup = np.zeros(len(close), dtype=int)
    sell_setup = np.zeros(len(close), dtype=int)

    c_vals = close.values
    l4_vals = lag4.values

    curr_buy = 0
    curr_sell = 0

    for i in range(4, len(c_vals)):
        # Low 9 (潛在買入)
        if c_vals[i] < l4_vals[i]:
            curr_buy += 1
        else:
            curr_buy = 0
        buy_setup[i] = curr_buy

        # High 9 (潛在賣出)
        if c_vals[i] > l4_vals[i]:
            curr_sell += 1
        else:
            curr_sell = 0
        sell_setup[i] = curr_sell

    # --- B. 檢查「今日」最新狀態 ---
    last_buy = buy_setup[-1]
    last_sell = sell_setup[-1]

    current_signal = "無"
    if last_buy >= 9:
        current_signal = f"🔴低{last_buy}"
    elif last_buy >= 1:
        current_signal = f"跌:{last_buy}"
    elif last_sell >= 9:
        current_signal = f"🟢高{last_sell}"
    elif last_sell >= 1:
        current_signal = f"升:{last_sell}"

    # --- C. 多時段歷史回測 (10, 30, 60日) ---
    # 儲存不同天數的勝負結果
    wins_buy = {10: [], 30: [], 60: []}
    wins_sell = {10: [], 30: [], 60: []}

    # 1. 回測買入訊號 (低 9)
    buy_indices = np.where(buy_setup[:-1] == 9)[0]
    for idx in buy_indices:
        entry = c_vals[idx]
        for d in [10, 30, 60]:
            if idx + d < len(c_vals):
                exit_price = c_vals[idx + d]
                # 買入後升 = Win
                wins_buy[d].append(1 if exit_price > entry else 0)

    # 2. 回測賣出訊號 (高 9)
    sell_indices = np.where(sell_setup[:-1] == 9)[0]
    for idx in sell_indices:
        entry = c_vals[idx]
        for d in [10, 30, 60]:
            if idx + d < len(c_vals):
                exit_price = c_vals[idx + d]
                # 賣出後跌 = Win
                wins_sell[d].append(1 if exit_price < entry else 0)

    # 計算勝率 Helper
    def calc(lst):
        return (sum(lst) / len(lst) * 100) if lst else 0

    return {
        "Ticker": ticker.replace("-USD", ""),
        "Price": round(c_vals[-1], 2),
        "Signal": current_signal,

        "Buy_Count": len(buy_indices),
        "Buy_10": calc(wins_buy[10]),
        "Buy_30": calc(wins_buy[30]),
        "Buy_60": calc(wins_buy[60]),

        "Sell_Count": len(sell_indices),
        "Sell_10": calc(wins_sell[10]),
        "Sell_30": calc(wins_sell[30]),
        "Sell_60": calc(wins_sell[60]),
    }

# ==========================================
# 3. 主程序
# ==========================================
print(f"正在下載 {target_tickers} 數據並進行多時段回測...\n")

data = yf.download(target_tickers, period="2y", interval="1d", group_by='ticker', progress=False)

# 設定顯示格式
header = f"{'資產':<8} {'現價':<8} {'訊號':<8} | {'低9買入勝率 (10/30/60日)':<28} | {'高9賣出勝率 (10/30/60日)':<28}"
print("-" * 95)
print(header)
print("-" * 95)

for ticker in target_tickers:
    try:
        # 處理數據結構
        if ticker in data.columns or (isinstance(data.columns, pd.MultiIndex) and ticker in data.columns.levels[0]):
             if isinstance(data.columns, pd.MultiIndex):
                 df = data[ticker].dropna()
             else:
                 df = data.dropna()
        else:
             continue

        if df.empty: continue

        r = analyze_stock(ticker, df)

        if r:
            # 格式化勝率字串
            b_stats = f"N={r['Buy_Count']:<2} {r['Buy_10']:.0f}%  {r['Buy_30']:.0f}%  {r['Buy_60']:.0f}%"
            s_stats = f"N={r['Sell_Count']:<2} {r['Sell_10']:.0f}%  {r['Sell_30']:.0f}%  {r['Sell_60']:.0f}%"

            # 加火把 🔥 (如果 30日勝率 > 65%)
            if r['Buy_30'] > 65: b_stats += "🔥"
            if r['Sell_30'] > 65: s_stats += "🔥"

            print(f"{r['Ticker']:<8} {r['Price']:<8} {r['Signal']:<8} | {b_stats:<28} | {s_stats:<28}")

    except Exception as e:
        pass

print("-" * 95)
print("註: N=出現次數 | 百分比代表持有該日數後的獲利機率 | 🔥=30日勝率>65%")

正在下載 ['VOO', 'QQQ', 'ETH-USD', 'BNB-USD', 'BTC-USD'] 數據並進行多時段回測...

-----------------------------------------------------------------------------------------------
資產       現價       訊號       | 低9買入勝率 (10/30/60日)           | 高9賣出勝率 (10/30/60日)          
-----------------------------------------------------------------------------------------------
VOO      685.55   升:2      | N=2  0%  50%  100%           | N=17 47%  13%  13%          
QQQ      717.54   升:2      | N=2  50%  50%  100%          | N=13 46%  50%  42%          
ETH      2095.02  跌:1      | N=6  60%  20%  40%           | N=9  44%  56%  62%          
BNB      655.43   升:4      | N=6  33%  67%  50%🔥          | N=11 45%  30%  30%          
BTC      76375.17 跌:3      | N=9  67%  33%  56%           | N=9  56%  56%  56%          
-----------------------------------------------------------------------------------------------
註: N=出現次數 | 百分比代表持有該日數後的獲利機率 | 🔥=30日勝率>65%


In [8]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# --- 設定區域 ---
SYMBOL = "BTC-USD"
MA_WINDOW = 50  # 策略核心：50週 MA
LAST_HALVING_DATE = "2024-04-20"  # 最近一次減半日期
CYCLE_TOP_DAYS = 530  # 預測見頂天數 (減半後)
CYCLE_BOTTOM_DAYS = 365 # 預測熊市持續天數 (見頂後)

def analyze_strategy():
    print(f"🔄 正在獲取 {SYMBOL} 數據並分析中... 請稍候...")

    # 1. 獲取數據 (週線)
    df = yf.download(SYMBOL, period="2y", interval="1wk", progress=False, auto_adjust=True)

    if df.empty:
        print("❌ 錯誤：無法獲取數據，請檢查網絡連接。")
        return

    # 2. 計算 50週移動平均線 (MA)
    df['50_MA'] = df['Close'].rolling(window=MA_WINDOW).mean()

    # 獲取最新一週的數據
    latest_data = df.iloc[-1]
    current_price = float(latest_data['Close'].iloc[0])
    current_ma = float(latest_data['50_MA'].iloc[0])

    # 3. 計算週期時間 (Time Cycle)
    last_halving = datetime.strptime(LAST_HALVING_DATE, "%Y-%m-%d")
    today = datetime.now()
    days_since_halving = (today - last_halving).days

    # 計算關鍵日期
    estimated_top_date = last_halving + timedelta(days=CYCLE_TOP_DAYS)
    estimated_bull_start_date = estimated_top_date + timedelta(days=CYCLE_BOTTOM_DAYS)

    # --- 輸出報告 ---
    print("\n" + "="*40)
    print(f"📊 泰瑞週期趨勢混合模型分析報告 - {today.strftime('%Y-%m-%d')}")
    print("="*40)

    print(f"\n1️⃣  【技術面 (50週 MA)】")
    print(f"   現價 (BTC): ${current_price:,.2f}")
    print(f"   50週 MA線:  ${current_ma:,.2f}")

    trend_status = ""
    if current_price > current_ma:
        diff = ((current_price - current_ma) / current_ma) * 100
        print(f"   ✅ 狀態: 價格在 MA 之上 ({diff:.2f}%) -> 牛市趨勢維持")
        trend_status = "BULL"
    else:
        diff = ((current_ma - current_price) / current_ma) * 100
        print(f"   ⚠️ 狀態: 價格跌穿 MA ({diff:.2f}%) -> 熊市/趨勢破壞警號")
        trend_status = "BEAR"

    print(f"\n2️⃣  【時間週期 (Time Cycle)】")
    print(f"   上次減半日: {LAST_HALVING_DATE}")
    print(f"   已過天數:   {days_since_halving} 天")
    print(f"   --------------------------------")
    print(f"   📅 預測本輪頂部: {estimated_top_date.strftime('%Y-%m-%d')} (減半+530日)")

    # 判斷頂部是否已過
    if today > estimated_top_date:
         print(f"      👉 狀態: [已過期] 理論上牛市已結束")
    else:
         days_left = (estimated_top_date - today).days
         print(f"      👉 狀態: [未到] 還有 {days_left} 天")

    print(f"   📅 預測下輪牛市: {estimated_bull_start_date.strftime('%Y-%m-%d')} (底部反轉日)")
    print(f"      👉 (即頂部後 365 天熊市結束之日)")

    cycle_status = ""
    if days_since_halving < 400:
        cycle_status = "SAFE"
    elif days_since_halving < CYCLE_TOP_DAYS:
        cycle_status = "WARNING"
    else:
        cycle_status = "DANGER"

    print(f"\n3️⃣  【綜合操作建議 (僅供參考)】")

    if trend_status == "BULL":
        if cycle_status == "SAFE":
            print("   🟢 建議: 【買入 / 持有 (HODL)】")
            print("   📝 理由: 趨勢向上且週期健康。")
        elif cycle_status == "WARNING":
            print("   🟡 建議: 【持有 / 準備止賺】")
            print("   📝 理由: 魚尾階段，切勿加大槓桿。")
        else: # DANGER
            print("   🟠 建議: 【分批止賺 / 嚴格止損】")
            print("   📝 理由: 時間到頂，雖然價格未崩，但隨時會變天。")

    else: # BEAR (價格 < 50MA)
        if cycle_status == "SAFE":
             print("   🟡 建議: 【觀望 / 等待收復】")
             print("   📝 理由: 週期良好但技術轉差，等待站回 50MA。")
        else:
             print("   🔴 建議: 【離場 / 現金為王】")
             print("   📝 理由: 跌穿生命線 + 週期結束。熊市確立。")
             print(f"   ⏳ 耐心等待至 {estimated_bull_start_date.strftime('%Y-%m-%d')} 附近再部署。")

    print("\n" + "-"*40)
    print("🔍 【蘇蘇執行清單 Checklist 提醒】")
    print("   [ ] 檢查 Whale Alert (巨鯨動向)")
    print("   [ ] 深呼吸，投資係長跑，現在是休息時間")
    print("   [ ] 記得今晚陪伴家人一小時")
    print("="*40 + "\n")

if __name__ == "__main__":
    analyze_strategy()

🔄 正在獲取 BTC-USD 數據並分析中... 請稍候...

📊 泰瑞週期趨勢混合模型分析報告 - 2026-05-24

1️⃣  【技術面 (50週 MA)】
   現價 (BTC): $76,375.17
   50週 MA線:  $94,029.17
   ⚠️ 狀態: 價格跌穿 MA (18.78%) -> 熊市/趨勢破壞警號

2️⃣  【時間週期 (Time Cycle)】
   上次減半日: 2024-04-20
   已過天數:   764 天
   --------------------------------
   📅 預測本輪頂部: 2025-10-02 (減半+530日)
      👉 狀態: [已過期] 理論上牛市已結束
   📅 預測下輪牛市: 2026-10-02 (底部反轉日)
      👉 (即頂部後 365 天熊市結束之日)

3️⃣  【綜合操作建議 (僅供參考)】
   🔴 建議: 【離場 / 現金為王】
   📝 理由: 跌穿生命線 + 週期結束。熊市確立。
   ⏳ 耐心等待至 2026-10-02 附近再部署。

----------------------------------------
🔍 【蘇蘇執行清單 Checklist 提醒】
   [ ] 檢查 Whale Alert (巨鯨動向)
   [ ] 深呼吸，投資係長跑，現在是休息時間
   [ ] 記得今晚陪伴家人一小時

